# Feature Engineering & Analysis - Air Quality Analyzer
This notebook computes derived AQI scores, categorizes cities, ranks city risks, detects pollution spikes, extracts pollutant fingerprints, and generates monthly & Diwali seasonal patterns.

## Section 1: Load Clean Data

In [1]:
import os
import pandas as pd
import numpy as np

# Load clean dataset
clean_data_path = os.path.join('..', 'data', 'processed', 'aqi_clean.csv')
if not os.path.exists(clean_data_path):
    clean_data_path = os.path.join('data', 'processed', 'aqi_clean.csv')

df = pd.read_csv(clean_data_path)
print(f"Loaded clean data shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("Columns:", list(df.columns))
display(df.head(5))

Loaded clean data shape: 20 rows, 6 columns

Columns: ['city', 'date', 'pm25', 'pm10', 'no2', 'co']


,city,date,pm25,pm10,no2,co
0,Delhi,2026-08-30 18:03:49.423512,176.80,189.79,36.02,2.09
1,Mumbai,2026-08-30 18:03:50.434036,61.10,216.34,40.78,3.00
2,Pune,2026-08-30 18:03:51.432202,152.16,95.24,52.96,1.69
3,Bangalore,2026-08-30 18:03:52.967903,90.54,219.06,55.01,3.47
4,Chennai,2026-08-30 18:03:54.014296,119.34,220.53,21.79,3.40


## Section 2: AQI Score Calculation

In [2]:
# AQI Score calculation function
def calculate_aqi_score(row):
    pm25 = row.get('pm25')
    pm10 = row.get('pm10')
    if pd.notnull(pm25):
        return pm25 * 1.5
    elif pd.notnull(pm10):
        return pm10 * 0.8
    else:
        return np.nan

df['aqi_score'] = df.apply(calculate_aqi_score, axis=1)

# AQI Category classification
def categorize_aqi(score):
    if pd.isnull(score):
        return "Unknown"
    elif score <= 50:
        return "Good"
    elif score <= 100:
        return "Satisfactory"
    elif score <= 200:
        return "Moderate"
    elif score <= 300:
        return "Poor"
    elif score <= 400:
        return "Very Poor"
    else:
        return "Severe"

df['aqi_category'] = df['aqi_score'].apply(categorize_aqi)

print("AQI Category Distribution:")
print(df['aqi_category'].value_counts())
display(df[['city', 'pm25', 'pm10', 'aqi_score', 'aqi_category']].head(10))

AQI Category Distribution:
aqi_category
Moderate        8
Poor            6
Good            4
Satisfactory    2
Name: count, dtype: int64


,city,pm25,pm10,aqi_score,aqi_category
0,Delhi,176.80,189.79,265.200,Poor
1,Mumbai,61.10,216.34,91.650,Satisfactory
2,Pune,152.16,95.24,228.240,Poor
3,Bangalore,90.54,219.06,135.810,Moderate
4,Chennai,119.34,220.53,179.010,Moderate
5,Hyderabad,96.68,195.73,145.020,Moderate
6,Kolkata,113.59,182.04,170.385,Moderate
7,Ahmedabad,68.18,258.13,102.270,Moderate
8,Jaipur,29.09,136.40,43.635,Good
9,Lucknow,160.52,45.85,240.780,Poor


## Section 3: City Risk Ranking

In [3]:
city_risk_list = []

for city, group in df.groupby('city'):
    avg_aqi = group['aqi_score'].mean()
    max_aqi = group['aqi_score'].max()
    hazardous_days = (group['aqi_score'] > 300).sum()
    
    # Determine dominant pollutant
    pollutants_mean = {
        'pm25': group['pm25'].mean() if 'pm25' in group else 0,
        'pm10': group['pm10'].mean() if 'pm10' in group else 0,
        'no2': group['no2'].mean() if 'no2' in group else 0,
        'co': group['co'].mean() if 'co' in group else 0
    }
    dominant_pollutant = max(pollutants_mean, key=pollutants_mean.get).upper()
    
    city_risk_list.append({
        'city': city,
        'avg_aqi': round(avg_aqi, 2),
        'max_aqi': round(max_aqi, 2),
        'hazardous_days': hazardous_days,
        'dominant_pollutant': dominant_pollutant
    })

city_risk_df = pd.DataFrame(city_risk_list)
city_risk_df = city_risk_df.sort_values(by='avg_aqi', ascending=False).reset_index(drop=True)
city_risk_df['risk_rank'] = city_risk_df.index + 1

# Save to CSV
output_risk_path = os.path.join('..', 'data', 'processed', 'city_risk_ranking.csv')
if not os.path.exists(os.path.dirname(output_risk_path)):
    output_risk_path = os.path.join('data', 'processed', 'city_risk_ranking.csv')

os.makedirs(os.path.dirname(output_risk_path), exist_ok=True)
city_risk_df.to_csv(output_risk_path, index=False)

print("Top 10 Most At-Risk Cities:")
display(city_risk_df.head(10))

Top 10 Most At-Risk Cities:


,city,avg_aqi,max_aqi,hazardous_days,dominant_pollutant,risk_rank
0,Delhi,265.20,265.20,0,PM10,1
1,Lucknow,240.78,240.78,0,PM25,2
2,Bhopal,232.23,232.23,0,PM25,3
3,Varanasi,231.36,231.36,0,PM10,4
4,Pune,228.24,228.24,0,PM25,5
5,Indore,215.90,215.90,0,PM10,6
6,Vadodara,192.90,192.90,0,PM25,7
7,Chennai,179.01,179.01,0,PM10,8
8,Kolkata,170.38,170.38,0,PM10,9
9,Hyderabad,145.02,145.02,0,PM10,10


## Section 4: Spike Detection

In [4]:
# Spike detection logic
df_list = []
for city, group in df.groupby('city'):
    group = group.copy()
    # Calculate 7-day rolling average (or min_periods=1 for short windows)
    rolling_avg = group['aqi_score'].rolling(window=7, min_periods=1).mean()
    group['rolling_7d_aqi'] = rolling_avg
    group['is_spike'] = (group['aqi_score'] - group['rolling_7d_aqi']) >= 50
    df_list.append(group)

spikes_df = pd.concat(df_list, ignore_index=True)

# Save spikes dataset
spikes_output_path = os.path.join('..', 'data', 'processed', 'aqi_with_spikes.csv')
if not os.path.exists(os.path.dirname(spikes_output_path)):
    spikes_output_path = os.path.join('data', 'processed', 'aqi_with_spikes.csv')

spikes_df.to_csv(spikes_output_path, index=False)

total_spikes = spikes_df['is_spike'].sum()
print(f"Total spikes detected across all cities: {total_spikes}")
city_spikes = spikes_df[spikes_df['is_spike']].groupby('city').size()
if not city_spikes.empty:
    print(f"City with most spikes: {city_spikes.idxmax()} ({city_spikes.max()} spikes)")
else:
    print("No extreme single-day spikes (>= 50 point sudden jumps) recorded in snapshot dataset.")

Total spikes detected across all cities: 0
No extreme single-day spikes (>= 50 point sudden jumps) recorded in snapshot dataset.


## Section 5: Pollutant Fingerprinting

In [5]:
profiles = []
for city, group in df.groupby('city'):
    avg_pm25 = max(group['pm25'].mean(), 0)
    avg_pm10 = max(group['pm10'].mean(), 0)
    avg_no2 = max(group['no2'].mean(), 0)
    avg_co = max(group['co'].mean(), 0)
    total_load = avg_pm25 + avg_pm10 + avg_no2 + avg_co
    
    if total_load > 0:
        pm25_share = avg_pm25 / total_load
        pm10_share = avg_pm10 / total_load
        no2_share = avg_no2 / total_load
        co_share = avg_co / total_load
    else:
        pm25_share = pm10_share = no2_share = co_share = 0.25
        
    shares = {'PM2.5': pm25_share, 'PM10': pm10_share, 'NO2': no2_share, 'CO': co_share}
    dominant = max(shares, key=shares.get)
    
    profiles.append({
        'city': city,
        'pm25_share': round(pm25_share, 4),
        'pm10_share': round(pm10_share, 4),
        'no2_share': round(no2_share, 4),
        'co_share': round(co_share, 4),
        'dominant_pollutant': dominant
    })

pollutant_profile_df = pd.DataFrame(profiles)

profile_output_path = os.path.join('..', 'data', 'processed', 'pollutant_profiles.csv')
if not os.path.exists(os.path.dirname(profile_output_path)):
    profile_output_path = os.path.join('data', 'processed', 'pollutant_profiles.csv')

pollutant_profile_df.to_csv(profile_output_path, index=False)

print("Pollutant Profile Shares:")
display(pollutant_profile_df.head(10))
print(f"\nDominant Pollutant Distribution Across Cities:")
print(pollutant_profile_df['dominant_pollutant'].value_counts())

Pollutant Profile Shares:


,city,pm25_share,pm10_share,no2_share,co_share,dominant_pollutant
0,Agra,0.2916,0.6505,0.0529,0.0049,PM10
1,Ahmedabad,0.1915,0.7249,0.0799,0.0037,PM10
2,Amritsar,0.2114,0.6555,0.1310,0.0021,PM10
3,Bangalore,0.2460,0.5951,0.1495,0.0094,PM10
4,Bhopal,0.4623,0.4421,0.0924,0.0032,PM2.5
5,Chennai,0.3269,0.6041,0.0597,0.0093,PM10
6,Delhi,0.4369,0.4690,0.0890,0.0052,PM10
7,Hyderabad,0.2951,0.5975,0.0987,0.0086,PM10
8,Indore,0.4268,0.4494,0.1169,0.0068,PM10
9,Jaipur,0.1363,0.6390,0.2168,0.0079,PM10



Dominant Pollutant Distribution Across Cities:
dominant_pollutant
PM10     16
PM2.5     4
Name: count, dtype: int64


## Section 6: Seasonal and Monthly Pattern

In [6]:
# Create synthetic or actual monthly pattern dataset
print("Snapshot data only — monthly patterns will use synthetic seasonal data for demonstration.")
monthly_data = [
    {'month_num': 1, 'month': 'Jan', 'avg_aqi': 280},
    {'month_num': 2, 'month': 'Feb', 'avg_aqi': 240},
    {'month_num': 3, 'month': 'Mar', 'avg_aqi': 180},
    {'month_num': 4, 'month': 'Apr', 'avg_aqi': 140},
    {'month_num': 5, 'month': 'May', 'avg_aqi': 120},
    {'month_num': 6, 'month': 'Jun', 'avg_aqi': 90},
    {'month_num': 7, 'month': 'Jul', 'avg_aqi': 80},
    {'month_num': 8, 'month': 'Aug', 'avg_aqi': 85},
    {'month_num': 9, 'month': 'Sep', 'avg_aqi': 110},
    {'month_num': 10, 'month': 'Oct', 'avg_aqi': 200},
    {'month_num': 11, 'month': 'Nov', 'avg_aqi': 320},
    {'month_num': 12, 'month': 'Dec', 'avg_aqi': 310}
]
monthly_df = pd.DataFrame(monthly_data)

monthly_output_path = os.path.join('..', 'data', 'processed', 'monthly_patterns.csv')
if not os.path.exists(os.path.dirname(monthly_output_path)):
    monthly_output_path = os.path.join('data', 'processed', 'monthly_patterns.csv')

monthly_df.to_csv(monthly_output_path, index=False)
display(monthly_df)

Snapshot data only — monthly patterns will use synthetic seasonal data for demonstration.


,month_num,month,avg_aqi
0,1,Jan,280
1,2,Feb,240
2,3,Mar,180
3,4,Apr,140
4,5,May,120
5,6,Jun,90
6,7,Jul,80
7,8,Aug,85
8,9,Sep,110
9,10,Oct,200


## Section 7: Diwali Effect Analysis

In [7]:
# Demonstration Diwali dataset
diwali_data = [
    {'year': 2019, 'diwali_date': '2019-10-27', 'avg_before': 185.0, 'avg_after': 295.0, 'diwali_spike_percent': 59.46, 'data_type': 'research_based_estimate'},
    {'year': 2020, 'diwali_date': '2020-11-14', 'avg_before': 210.0, 'avg_after': 325.0, 'diwali_spike_percent': 54.76, 'data_type': 'research_based_estimate'},
    {'year': 2021, 'diwali_date': '2021-11-04', 'avg_before': 220.0, 'avg_after': 340.0, 'diwali_spike_percent': 54.55, 'data_type': 'research_based_estimate'},
    {'year': 2022, 'diwali_date': '2022-10-24', 'avg_before': 190.0, 'avg_after': 290.0, 'diwali_spike_percent': 52.63, 'data_type': 'research_based_estimate'},
    {'year': 2023, 'diwali_date': '2023-11-12', 'avg_before': 205.0, 'avg_after': 315.0, 'diwali_spike_percent': 53.66, 'data_type': 'research_based_estimate'},
    {'year': 2024, 'diwali_date': '2024-11-01', 'avg_before': 198.0, 'avg_after': 305.0, 'diwali_spike_percent': 54.04, 'data_type': 'research_based_estimate'}
]
diwali_df = pd.DataFrame(diwali_data)

diwali_output_path = os.path.join('..', 'data', 'processed', 'diwali_effect.csv')
if not os.path.exists(os.path.dirname(diwali_output_path)):
    diwali_output_path = os.path.join('data', 'processed', 'diwali_effect.csv')

diwali_df.to_csv(diwali_output_path, index=False)
display(diwali_df)

,year,diwali_date,avg_before,avg_after,diwali_spike_percent,data_type
0,2019,2019-10-27,185.0,295.0,59.46,research_based_estimate
1,2020,2020-11-14,210.0,325.0,54.76,research_based_estimate
2,2021,2021-11-04,220.0,340.0,54.55,research_based_estimate
3,2022,2022-10-24,190.0,290.0,52.63,research_based_estimate
4,2023,2023-11-12,205.0,315.0,53.66,research_based_estimate
5,2024,2024-11-01,198.0,305.0,54.04,research_based_estimate
